In [1]:
import os
import psutil

# Get the current process
# p = psutil.Process(os.getpid())

# # Set the process to run only on the first CPU core
# p.cpu_affinity([76])

In [2]:
import sys, os

import numpy as np
from umap import UMAP
import time
import torch
import matplotlib.pyplot as plt
import pandas as pd  
import scanpy as sc
import scipy.sparse as sps
import scipy.io as sio
# import scipy.sparse as sp
from os.path import join
import h5py
import warnings
import gc

import scmomat 

plt.rcParams["font.size"] = 10

/home/xuhua/xuhua_disco/miniforge3/envs/spatialdata/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def wrap_warn_plot(adata, basis, color, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        sc.pl.embedding(adata, basis=basis, color=color, **kwargs)

def get_umap(ad, use_reps=[]):
    for use_rep in use_reps:
        umap_add_key = f'{use_rep}_umap'
        sc.pp.neighbors(ad, use_rep=use_rep, n_neighbors=15)
        sc.tl.umap(ad)
        ad.obsm[umap_add_key] = ad.obsm['X_umap']
    return ad

def split_ob(ads, ad_ref, ob='obs', key='emb2'):
    len_ads = [_.n_obs for _ in ads]
    if ob=='obsm':
        split_obsms = np.split(ad_ref.obsm[key], np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obsms):
            ad.obsm[key] = v
    else:
        split_obs = np.split(ad_ref.obs[key].to_list(), np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obs):
            ad.obs[key] = v


In [5]:
from batchCorr import HARMONY

In [10]:
root_data_dir = '../../../data/processed/Simulations/'

interval = 1000
batch_size = 0.1
seed = 0
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

K = 50
lamb = 0.01
T = 2000
lr = 0.01
n_neighbors = 40

for zero_p in ['0.1', '0.2', '0.3', '0.4', '0.5']:   # ['0.1', '0.2', '0.3', '0.4', '0.5']
    data_dirs = os.listdir(root_data_dir)
    foldn = list(filter(lambda x: x.startswith(f'rna-{zero_p}'), data_dirs))[0]
    data_dir = root_data_dir + foldn

    for seed in [1, 2, 3]:
        input_train_mod1 = sc.read_h5ad(join(data_dir, f'{seed}/ad_rna1.h5ad'))
        input_train_mod2 = sc.read_h5ad(join(data_dir, f'{seed}/ad_adt1.h5ad'))
        input_test_mod1 = sc.read_h5ad(join(data_dir, f'{seed}/ad_rna2.h5ad'))
        input_test_mod2 = sc.read_h5ad(join(data_dir, f'{seed}/ad_adt2.h5ad'))

        input_test_mod1.obs_names = [f"test-mod1-{i}" for i in range(input_test_mod1.shape[0])]
        input_test_mod2.obs_names = [f"test-mod2-{i}" for i in range(input_test_mod2.shape[0])]

        input_train_mod1.obs['batch'] = 's1'
        input_train_mod2.obs['batch'] = 's1'
        input_test_mod1.obs['batch'] = 's2'
        input_test_mod2.obs['batch'] = 's3'

        hvg_names = np.array(input_train_mod1.var_names)
        hvp_names = np.array(input_train_mod2.var_names.to_list())

        # READ IN THE COUNT MATRICES
        counts_rnas = []
        for adi in [input_train_mod1, input_test_mod1, None]:
            if adi is not None:
                count = scmomat.preprocess(adi.layers['counts'], modality = "RNA", log = False)
                counts_rnas.append(count)
            else:
                counts_rnas.append(None)

        counts_atacs = []
        for adi in [input_train_mod2, None, input_test_mod2]:
            if adi is not None:
                count = scmomat.preprocess(adi.layers['counts'], modality = "ADT", log = True)
                counts_atacs.append(count)
            else:
                counts_atacs.append(None)

        # CREATE THE COUNTS OBJECT
        n_batches = 3
        feats_name = {"rna": hvg_names, "atac": hvp_names}
        counts = {"feats_name": feats_name, "nbatches": n_batches, "rna":counts_rnas, "atac": counts_atacs}

        # read in the ground truth labels
        # NOTE: Plot the result before post-processing
        labels = [input_train_mod1.obs['gt'], input_test_mod1.obs['gt'], input_test_mod2.obs['gt']]
        batches = ['bridge']*input_train_mod1.shape[0] + ['test-rna']*input_test_mod1.shape[0] + ['test-atac']*input_test_mod2.shape[0]

        start_time = time.time()
        model = scmomat.scmomat_model(counts = counts, K = K, batch_size = batch_size, interval = interval, lr = lr, 
                                      lamb = lamb, seed = seed, device = device)
        losses = model.train_func(T = T)
        end_time = time.time()
        print("running time: " + str(end_time - start_time))

        zs = model.extract_cell_factors()
        r = None
        knn_indices, knn_dists = scmomat.calc_post_graph(zs, n_neighbors, njobs = 1, r = r)

        ad_mosaic = sc.AnnData(np.vstack(zs), obsm={"X_emb":np.vstack(zs)})
        ad_mosaic.obs['gt'] = np.hstack(labels)
        ad_mosaic.obs['batch'] = np.hstack(batches)

        ad_mosaic.obsm['X_emb_har'] = HARMONY(pd.DataFrame(ad_mosaic.obsm['X_emb']), ad_mosaic.obs['batch'].to_list())
        split_ob([input_train_mod1, input_test_mod1, input_test_mod2], ad_mosaic, ob='obsm', key='X_emb_har')
        
        df_emb = pd.DataFrame(ad_mosaic.obsm['X_emb_har'], 
                    index=input_train_mod1.obs_names.to_list() + input_test_mod1.obs_names.to_list() + input_test_mod2.obs_names.to_list())
        
        os.makedirs(f'../../../results/embeddings/scMoMaT/Simulations/{foldn}/{seed}', exist_ok=True)
        df_emb.to_csv(f'../../../results/embeddings/scMoMaT/Simulations/{foldn}/{seed}/df_emb.csv')
       

Input sanity check...
Finished.
Epoch 1, Validating Loss: 149.7506
	 loss reconstruction: 0.13975
	 loss regularization: 1.00000
Epoch 1000, Validating Loss: 89.2570
	 loss reconstruction: 0.08908
	 loss regularization: 0.01774
running time: 34.51890826225281
calculating pairwise distance, time used 0.1938s
knn separate, time used 0.2484s
modify distance, time used 0.0036s
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
Reach convergence after 3 iteration(s).
Input sanity check...
Finished.
Epoch 1, Validating Loss: 149.1293
	 loss reconstruction: 0.13913
	 loss regularization: 1.00000
Epoch 1000, Validating Loss: 85.9031
	 loss reconstruction: 0.08575
	 loss regularization: 0.01520
running time: 33.351094245910645
calculating pairwise distance, time used 0.4392s
knn separate, time used 0.2482s
modify distance, time used 0.0049s
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 itera

In [7]:
out_dir

'/home/xuhua/xuhua_disco/gitrepo/BridgeNorm/figures/case10-up-down-maskP=0.4-be/'